In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = "deepseek-r1-distill-llama-70b"
llm = ChatGroq(model = model)

In [ ]:
import operator
from typing import List
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.prompts import PromptTemplate
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

def call_model(state: MessagesState):
    '''
    The function takes message state, fetches current message and invokes with the help of llm model.
    '''
    message = state["messages"][-1] ## Getting current message
    response = llm.invoke(message)

    return {"messages": [response]}

# state = {"messages": ["Hello llm"]}
# call_model(state)

In [31]:
type(state["messages"][-1])

str

In [37]:
state = {"messages": [(HumanMessage,"Hello llm")]}

workflow = StateGraph(MessagesState)

workflow.add_node("LLM", call_model)

workflow.add_edge(START, "LLM")
workflow.add_edge("LLM", END)


app = workflow.compile()

response = app.invoke(state)
response

ValueError: Unexpected message type: '<class 'langchain_core.messages.human.HumanMessage'>'. Use one of 'human', 'user', 'ai', 'assistant', 'function', 'tool', 'system', or 'developer'.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/MESSAGE_COERCION_FAILURE 

In [1]:
@tool
def search(query: str):
    '''
    This is my custom tool for searching a weather.
    '''
    if 'delhi' in query.lower():
        print("delhi")
    print("other")

NameError: name 'tool' is not defined

In [ ]:
tools = [search]

In [ ]:
llm_with_tool = llm.bind_tools(tools)

In [ ]:
response = llm_with_tool.invoke("what is a weather in delhi?")

In [ ]:
response

In [ ]:
response.tool_calls

In [ ]:
def call_model(state: MessagesState):
    message = state['messages']
    response = llm_with_tool.invoke(message)
    
    return {'messages': [response]}

In [ ]:
state = {"messages": [(HumanMessage,"Hello llm")]}
response = call_model(input) 

In [ ]:
def router(state: MessagesState):
    message = state[messages][-1]

    if message.tool_calls:
        return 'tools'
    return END

In [ ]:
tool_node = ToolNode(tools)

In [2]:
workflow2 = StateGraph(MessagesState)

workflow2.add_node('llmwithtools', call_model)
workflow2.add_node('tools', tool_node)

workflow2.add_edge(START, "llmwithtools")
workflow2.add_edge("llmwithtools", 
                   router,
                   {"tools": "tools",
                    END: END
                   })

app = workflow.compile()

NameError: name 'StateGraph' is not defined